# Error Analysis (Klasifikasi + Retrieval)
Notebook ini membedah **di mana dan kenapa model salah**, sebagai bahan pembahasan BAB IV. Dua bagian:
- **Klasifikasi**: false positive/negative, diagnosis dimensi terlemah, deteksi pola *containment* & *semantik-terjemahan*.
- **Retrieval**: distribusi peringkat target, kasus peringkat rendah/tak-terjangkau, distraktor yang mengalahkan target.

Membaca output Langkah D–F: `pasangan_berlabel_fitur.csv`, `ringkasan_hasil.csv` (bobot & threshold final), `gold_cases_categorized.csv`, `haystack_preprocessed_fonID.csv`, `haystack_emb.npy`, `haystack_emb_index.csv`, dan model semantik di Drive.


## 0. Setup & muat

In [ ]:
!pip install rapidfuzz metaphone fasttext-wheel -q
import os, numpy as np, pandas as pd
from rapidfuzz import process
from rapidfuzz.distance import JaroWinkler
from metaphone import doublemetaphone
pd.set_option("display.max_colwidth", 40)
print("Setup selesai.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 18.3 MB/s eta 0:00:00
Setup selesai.


In [ ]:
butuh = ["pasangan_berlabel_fitur.csv","ringkasan_hasil.csv","gold_cases_categorized.csv",
         "haystack_preprocessed_fonID.csv","haystack_emb.npy","haystack_emb_index.csv"]
if not all(os.path.exists(f) for f in butuh):
    from google.colab import files
    print("Upload output Langkah D–F & C:", [f for f in butuh if not os.path.exists(f)])
    files.upload()

pairs = pd.read_csv("pasangan_berlabel_fitur.csv").fillna("")
ringk = pd.read_csv("ringkasan_hasil.csv").iloc[0]
gold  = pd.read_csv("gold_cases_categorized.csv", dtype=str).fillna("")
hay   = pd.read_csv("haystack_preprocessed_fonID.csv", dtype=str).fillna("")
hay   = hay[hay["nama_base"].str.len()>0].reset_index(drop=True)
emb_index = pd.read_csv("haystack_emb_index.csv", dtype=str).fillna("")
emb   = np.load("haystack_emb.npy")

W = (float(ringk["bobot_JW"]), float(ringk["bobot_fonetik"]), float(ringk["bobot_semantik"]))
TH = float(ringk["threshold"])
print(f"Bobot final: JW={W[0]} Fonetik={W[1]} Semantik={W[2]} | Threshold={TH}")

Upload output Langkah D–F & C: ['haystack_preprocessed_fonID.csv']


Saving haystack_preprocessed_fonID.csv to haystack_preprocessed_fonID.csv
Bobot final: JW=0.5 Fonetik=0.1 Semantik=0.4 | Threshold=0.55


## 1. Muat model semantik (untuk skor query di retrieval)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SEM_DIM = emb.shape[1]
sem_model = None
try:
    import fasttext
    sem_model = fasttext.load_model("/content/drive/MyDrive/skripsi_merek/cc.id.100.bin")
    print("Model semantik dimuat.")
except Exception as e:
    print("Model semantik tak dimuat (skor semantik query = 0):", repr(e))

def sem_vec(t):
    if sem_model is None or not str(t).strip(): return None
    return sem_model.get_sentence_vector(str(t).replace("\n"," "))

Mounted at /content/drive


Model semantik dimuat.


## 2. Analisis error KLASIFIKASI
Skor pasangan dihitung ulang dgn bobot final, lalu dibandingkan label. Fokus pada **false negative** (mirip tapi lolos — paling berbahaya di pemeriksaan merek).


In [ ]:
pairs["skor"] = W[0]*pairs["jw"] + W[1]*pairs["phon"] + W[2]*pairs["sem"]
pairs["pred"] = (pairs["skor"] >= TH).astype(int)

# fokus pasangan hard (dari gold), abaikan easy-negatif sampling utk analisis kualitatif
hard = pairs[pairs["jenis"]=="hard"].copy()
fp = hard[(hard["y"]==0) & (hard["pred"]==1)]
fn = hard[(hard["y"]==1) & (hard["pred"]==0)]
print(f"Pada pasangan hard (gold): FP={len(fp)}  FN={len(fn)}")

print("\n=== FALSE NEGATIVE (mirip tapi lolos) ===")
print(fn[["a","b","jw","phon","sem","skor"]].round(3).to_string(index=False))
print("\n=== FALSE POSITIVE (tidak mirip tapi ditandai) ===")
print(fp[["a","b","jw","phon","sem","skor"]].round(3).to_string(index=False))

Pada pasangan hard (gold): FP=4  FN=4

=== FALSE NEGATIVE (mirip tapi lolos) ===
                a            b    jw  phon    sem  skor
polobyralphlauren r l p c polo 0.507 0.730  0.120 0.374
               gs          gsj 0.911 0.911 -0.027 0.536
   safety joggers        joger 0.424 0.492  0.553 0.482
    purekids baby      my baby 0.407 0.492  0.680 0.525

=== FALSE POSITIVE (tidak mirip tapi ditandai) ===
                  a                       b    jw  phon   sem  skor
  magal mapogalmegi                galmaegi 0.590 0.731 0.728 0.659
people s place cafe       the people s cafe 0.757 0.841 0.936 0.837
             purebb              my pure bb 0.867 0.917 0.361 0.669
           imperial american pillo imperial 0.658 0.791 0.728 0.699


### 2a. Diagnosis: dimensi mana yang menyebabkan kesalahan?

In [ ]:
def diagnosa(row):
    dims = {"jw":row["jw"], "phon":row["phon"], "sem":row["sem"]}
    terlemah = min(dims, key=dims.get)
    terkuat  = max(dims, key=dims.get)
    return pd.Series({"dim_terlemah":terlemah, "nilai_terlemah":round(dims[terlemah],2),
                      "dim_terkuat":terkuat, "nilai_terkuat":round(dims[terkuat],2)})

if len(fn):
    print("Diagnosis FALSE NEGATIVE:")
    print(fn[["a","b"]].join(fn.apply(diagnosa,axis=1)).to_string(index=False))
    print("\nDistribusi dimensi terlemah pada FN:")
    print(fn.apply(diagnosa,axis=1)["dim_terlemah"].value_counts().to_string())

Diagnosis FALSE NEGATIVE:
                a            b dim_terlemah  nilai_terlemah dim_terkuat  nilai_terkuat
polobyralphlauren r l p c polo          sem            0.12        phon           0.73
               gs          gsj          sem           -0.03          jw           0.91
   safety joggers        joger           jw            0.42         sem           0.55
    purekids baby      my baby           jw            0.41         sem           0.68

Distribusi dimensi terlemah pada FN:
dim_terlemah
sem    2
jw     2


### 2b. Deteksi pola sistematis: *containment* & *semantik-terjemahan*
- **Containment**: satu nama memuat token kunci nama lain + kata tambahan (mis. "CROCODILE" ⊂ "CHILLINGTON CROCODILE"). Perbandingan string utuh cenderung gagal di sini.
- **Semantik-terjemahan**: JW & fonetik dua-duanya rendah, tapi label mirip → kemiripannya di makna (mis. "MAWAR" vs "ROSE"). Hanya semantik yang bisa menangkap.


In [ ]:
def token_containment(a, b):
    ta, tb = set(a.split()), set(b.split())
    if not ta or not tb: return False
    kecil, besar = (ta, tb) if len(ta)<=len(tb) else (tb, ta)
    return kecil.issubset(besar) and kecil != besar

def klasifikasi_pola(row):
    if token_containment(row["a"], row["b"]):
        return "containment"
    if row["y"]==1 and row["jw"]<0.6 and row["phon"]<0.6:
        return "semantik/terjemahan"
    return "lainnya"

fn2 = fn.copy()
fn2["pola"] = fn2.apply(klasifikasi_pola, axis=1)
print("Pola pada FALSE NEGATIVE:")
print(fn2[["a","b","pola"]].to_string(index=False))
print("\nRingkasan pola FN:")
print(fn2["pola"].value_counts().to_string())
print("\nCatatan BAB IV: pola 'containment' & 'semantik' adalah keterbatasan sistematis")
print("yang layak dibahas — bukan error acak. 'semantik/terjemahan' membenarkan adanya dimensi semantik.")
fn2.to_csv("error_false_negative.csv", index=False, encoding="utf-8-sig")
fp.to_csv("error_false_positive.csv", index=False, encoding="utf-8-sig")

Pola pada FALSE NEGATIVE:
                a            b                pola
polobyralphlauren r l p c polo             lainnya
               gs          gsj             lainnya
   safety joggers        joger semantik/terjemahan
    purekids baby      my baby semantik/terjemahan

Ringkasan pola FN:
pola
lainnya                2
semantik/terjemahan    2

Catatan BAB IV: pola 'containment' & 'semantik' adalah keterbatasan sistematis
yang layak dibahas — bukan error acak. 'semantik/terjemahan' membenarkan adanya dimensi semantik.


## 3. Analisis error RETRIEVAL
Peringkat target untuk tiap query `mirip` dihitung ulang. Fokus: query dengan target peringkat rendah / tak-terjangkau, dan distraktor yang mengalahkan target.


In [ ]:
gold_eval = gold[gold["kategori_eval"]=="bermakna"].reset_index(drop=True)
mirip = gold_eval[gold_eval["label_usulan"]=="mirip"].reset_index(drop=True)

cand_base = hay["nama_base"].tolist()
cand_dm   = hay["nama_phon"].apply(lambda t: "".join(doublemetaphone(x)[0] for x in t.split())).tolist()
id2row = {rid:i for i,rid in enumerate(emb_index["id"].tolist())}
cand_emb = np.vstack([emb[id2row[i]] if i in id2row else np.zeros(SEM_DIM) for i in hay["id"]]).astype("float32")
cand_emb_n = cand_emb/(np.linalg.norm(cand_emb,axis=1,keepdims=True)+1e-9)

def skor_semua(q_base, q_phon):
    qdm = "".join(doublemetaphone(x)[0] for x in q_phon.split())
    jw = process.cdist([q_base], cand_base, scorer=JaroWinkler.normalized_similarity)[0]
    ph = process.cdist([qdm], cand_dm, scorer=JaroWinkler.normalized_similarity)[0]
    qv = sem_vec(q_base)
    se = (cand_emb_n @ (qv/(np.linalg.norm(qv)+1e-9))) if qv is not None else np.zeros(len(cand_base))
    return W[0]*jw + W[1]*ph + W[2]*se

hasil=[]
for _, r in mirip.iterrows():
    sc = skor_semua(r["merek_a_base"], r["merek_a_phon"])
    order = np.argsort(-sc)
    rank=None
    for k,i in enumerate(order,1):
        if cand_base[i]==r["merek_b_base"] and cand_base[i]!=r["merek_a_base"]:
            rank=k; break
    hasil.append({"query":r["merek_a"], "target":r["merek_b"],
                  "rank": rank if rank else -1,
                  "terjangkau": rank is not None})
ret = pd.DataFrame(hasil)
terj = ret[ret["terjangkau"]]
print(f"Query mirip: {len(ret)} | terjangkau: {len(terj)} | tak-terjangkau: {(~ret['terjangkau']).sum()}")
print("\nDistribusi peringkat target (yg terjangkau):")
for k in [1,5,10,20,50]:
    print(f"  <= {k:>3}: {(terj['rank']<=k).mean():.3f}")
print(f"  Median rank: {terj['rank'].median():.0f}  |  Rank terburuk: {terj['rank'].max():.0f}")
ret.to_csv("error_retrieval_ranks.csv", index=False, encoding="utf-8-sig")

Query mirip: 47 | terjangkau: 39 | tak-terjangkau: 8

Distribusi peringkat target (yg terjangkau):
  <=   1: 0.026
  <=   5: 0.564
  <=  10: 0.615
  <=  20: 0.667
  <=  50: 0.769
  Median rank: 3  |  Rank terburuk: 10671


### 3a. Kasus peringkat rendah & tak-terjangkau

In [ ]:
print("=== Target TAK-TERJANGKAU (tidak ada di haystack / ejaan beda jauh) ===")
print(ret[~ret["terjangkau"]][["query","target"]].to_string(index=False))
print("\n=== Target peringkat TERBURUK (>10) ===")
print(terj[terj["rank"]>10].sort_values("rank",ascending=False)[["query","target","rank"]].to_string(index=False))

=== Target TAK-TERJANGKAU (tidak ada di haystack / ejaan beda jauh) ===
                     query                                                target
         EV; ELECTRO-VOICE                                       EV ELECTROVOICE
ROLLING STONES; THE STONES                                   STONES; STONES & CO
                 MATSUNAGA                              PRO MATSUNAGA; MATSUNAGA
       Mawar Super Loundry                                   Mawar Super Laundry
       STAR & Logo Bintang                       BLUE STAR; GREEN STAR; RED STAR
           CAP MAWAR MERAH                                        ROSE; RED ROSE
                      Wahl Wahl Erope; Wahl Europe; Wahl Ionic; Wahl By Sunshine
                 MATSUNAGA                             PRO MATSUNAGA / MATSUNAGA

=== Target peringkat TERBURUK (>10) ===
                query                                            target  rank
    POLOBYRALPHLAUREN                                     R.L.P.C. POLO 10671
  

### 3b. Distraktor: apa yang mengalahkan target pada kasus peringkat rendah?
Untuk beberapa query dengan target peringkat tinggi (>10), lihat merek apa yang menempati atas — memahami kenapa sistem 'salah fokus'.


In [ ]:
kasus = terj[terj["rank"]>10].head(3)
for _, kr in kasus.iterrows():
    qrow = mirip[mirip["merek_a"]==kr["query"]].iloc[0]
    sc = skor_semua(qrow["merek_a_base"], qrow["merek_a_phon"])
    order = np.argsort(-sc)[:5]
    print(f"\nQUERY: {kr['query']}  (target sebenarnya: {kr['target']} @rank {kr['rank']})")
    print("  Top-5 yang dikembalikan sistem:")
    for i in order:
        print(f"    {sc[i]:.3f}  {hay.iloc[i]['nama_merek'][:45]}")


QUERY: PROVIT-C  (target sebenarnya: PRO-FIT @rank 34)
  Top-5 yang dikembalikan sistem:
    1.000  PROVIT-C
    0.676  PRIORITA + LOGO P
    0.664  CRNVL + logo C
    0.664  crnvl + logo C
    0.664  D-POINT

QUERY: POLOBYRALPHLAUREN  (target sebenarnya: R.L.P.C. POLO @rank 10671)
  Top-5 yang dikembalikan sistem:
    0.965  POLORALPHLAUREN
    0.965  POLORALPHLAUREN
    0.965  POLORALPHLAUREN
    0.804  POLO RALPH LAUREN +LOGO
    0.804  POLO RALPH LAUREN

QUERY: HOREKAMALL  (target sebenarnya: HOREKA EXTRAVAGANZA @rank 20)
  Top-5 yang dikembalikan sistem:
    1.000  HOREKAMALL
    0.672  holaa
    0.667  PALLMALL + LOGO
    0.647  HOKAH
    0.645  The Farmhill


## 4. (Ringkas) Kasus `reputasi_dominan` yang dikecualikan
Kasus yang dikeluarkan dari metrik utama karena bergantung pada keterkenalan merek, bukan kemiripan tekstual/fonetik/semantik. Ditampilkan skornya untuk menunjukkan **kenapa** model tidak cocok menilainya.


In [ ]:
rep = gold[gold["kategori_eval"]=="dikecualikan_reputasi"].copy()
if len(rep):
    from rapidfuzz.distance import JaroWinkler as JW
    def jw(a,b): return JW.normalized_similarity(a,b) if a and b else 0.0
    rep["jw"] = rep.apply(lambda r: round(jw(r["merek_a_base"],r["merek_b_base"]),2), axis=1)
    print("Kasus reputasi_dominan (skor tekstual rendah tapi diputus mirip krn reputasi):")
    print(rep[["merek_a","merek_b","jw"]].to_string(index=False))
    print("\nCatatan BAB IV: skor tekstual rendah membuktikan model berbasis kemiripan")
    print("string/fonetik/semantik memang tak dirancang menangkap perlindungan merek terkenal.")
else:
    print("Tidak ada kasus reputasi_dominan.")

Kasus reputasi_dominan (skor tekstual rendah tapi diputus mirip krn reputasi):
     merek_a          merek_b   jw
    JOLLIBEE           JOLIBI 0.87
   HUGO BOSS HUGO SELECT LINE 0.68
          PB               PB 1.00
DIESEL HOUSE           DIESEL 0.90

Catatan BAB IV: skor tekstual rendah membuktikan model berbasis kemiripan
string/fonetik/semantik memang tak dirancang menangkap perlindungan merek terkenal.


## 5. Simpan & ringkasan untuk BAB IV
File error yang tersimpan: `error_false_negative.csv`, `error_false_positive.csv`, `error_retrieval_ranks.csv`.

**Poin pembahasan yang bisa langsung kamu tulis:**
1. FN didominasi pola *containment* (nama panjang memuat merek lain) & *semantik* → keterbatasan sistematis, bukan acak.
2. Kasus *semantik/terjemahan* (mawar↔rose) membenarkan dimensi semantik meski bobotnya kecil.
3. Target tak-terjangkau = keterbatasan cakupan haystack (jujur dilaporkan).
4. Kasus reputasi_dominan menegaskan batas ruang lingkup model (bukan penilai merek terkenal).
